In [3]:
import pandas as pd

In [5]:
def parse_line(line):
    parts = line.partition("\t")
    return parts[0].strip(), parts[2].strip()

file_dir = "./data/FDA/FDA_ZIP_folder_drugs@fda/"
file_name = "Products.txt"
file_path = file_dir + file_name

with open(file_path, "r") as f:
    lines = f.readlines()

data = []
for line in lines[1:]: # skip header
    appl_number, appl_info = parse_line(line)
    product_no, _ = parse_line(appl_info)
    data.append((appl_number, product_no))

df = pd.DataFrame(data, columns=["Application_number", "Product_number"])
df["Application_ID"] = df["Application_number"].astype(str) + "_" + df["Product_number"].astype(str)
df.to_csv(file_dir + "Products.csv", index=False)


In [6]:
df.describe()

,Application_number,Product_number,Application_ID
count,49405,49405,49405
unique,27890,54,49405
top,019630,001,000004_004
freq,53,27131,1


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49405 entries, 0 to 49404
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Application_number  49405 non-null  object
 1   Product_number      49405 non-null  object
 2   Application_ID      49405 non-null  object
dtypes: object(3)
memory usage: 1.1+ MB


-----
# Scrape indications

In [8]:
import requests
import json

In [17]:
with open("data/FDA/formatted_output_openFDA.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(len(df))

unique_drug_names = df["Drug name"].unique()
unique_drug_names = [name.strip() for name in unique_drug_names if isinstance(name, str)]
len(unique_drug_names)

unique_marketing_authorisation_numbers = df["Marketing authorisation Number"].unique()
len(unique_marketing_authorisation_numbers)

28288


28288

In [ ]:
def search_drug_by_brand(brand):
    url = "https://api.fda.gov/drug/label.json"
    params = {
        "search": f'openfda.brand_name:"{brand}"',
        # "limit": 1
    }

    response = requests.get(url, params=params)
    try:
        response.raise_for_status()
        data = response.json()
        if data["results"]:
            for result in data["results"]:
                print(result, "\n")

            indications = data["results"][0].get("indications_and_usage", ["No indication found."])
            print(f"\nBrand: {brand}\nIndication:\n{indications[0]}")
    except requests.exceptions.HTTPError as err:
        print(f"HTTP error: {err}")
    except Exception as e:
        print(f"Error: {e}")

    with open("result.txt", "w") as f:
        f.write(str(result))

    return result

for drug in unique_drug_names[:4]:
        search_drug_by_brand(drug)

result = search_drug_by_brand("TIZANIDINE HYDROCHLORIDE")


{'spl_product_data_elements': ['CAMILA NORETHINDRONE NORETHINDRONE NORETHINDRONE STARCH, CORN FD&C RED NO. 40 LACTOSE MONOHYDRATE MAGNESIUM STEARATE POVIDONE K90 SODIUM STARCH GLYCOLATE TYPE A POTATO Light Pink m;884 Chemical structure Figure 1 Image PDP-0.35mg table blister pack carton'], 'spl_unclassified_section': ['Rx Only Rev 11/2023 Patients should be counseled that oral contraceptives do not protect against transmission of HIV (AIDS) and other sexually transmitted diseases (STDs) such as Chlamydia, genital herpes, genital warts, gonorrhea, hepatitis B, and syphilis.'], 'description': ['DESCRIPTION Each light pink Camila® tablet provides a continuous oral contraceptive regimen of 0.35 mg norethindrone, USP daily, and has the following inactive ingredients: corn starch, FD&C red no. 40 aluminum lake, lactose monohydrate, magnesium stearate, povidone and sodium starch glycolate. The chemical name for norethindrone is 17-Hydroxy-19-nor-17α-pregn-4-en-20-yn-3-one. The structural form

{'spl_product_data_elements': ['tizanidine hydrochloride tizanidine hydrochloride TIZANIDINE HYDROCHLORIDE TIZANIDINE CELLULOSE, MICROCRYSTALLINE CROSCARMELLOSE SODIUM FD&C BLUE NO. 1 FD&C RED NO. 3 FERROSOFERRIC OXIDE GELATIN HYPROMELLOSE 2910 (5 MPA.S) LACTOSE MONOHYDRATE POTASSIUM HYDROXIDE SHELLAC SILICON DIOXIDE STEARIC ACID TITANIUM DIOXIDE light blue cap light blue body 1111 tizanidine hydrochloride tizanidine hydrochloride TIZANIDINE HYDROCHLORIDE TIZANIDINE CELLULOSE, MICROCRYSTALLINE CROSCARMELLOSE SODIUM FD&C BLUE NO. 1 FD&C RED NO. 3 GELATIN HYPROMELLOSE 2910 (5 MPA.S) LACTOSE MONOHYDRATE POTASSIUM HYDROXIDE SHELLAC SILICON DIOXIDE STEARIC ACID TITANIUM DIOXIDE violet cap white body 1112 tizanidine hydrochloride tizanidine hydrochloride TIZANIDINE HYDROCHLORIDE TIZANIDINE CELLULOSE, MICROCRYSTALLINE CROSCARMELLOSE SODIUM FD&C BLUE NO. 1 FD&C RED NO. 3 GELATIN HYPROMELLOSE 2910 (5 MPA.S) LACTOSE MONOHYDRATE POTASSIUM HYDROXIDE SHELLAC SILICON DIOXIDE STEARIC ACID TITANIUM DI